## Task #1: Image Sampling and Quantization

In [10]:
!pip install -q opencv-python matplotlib numpy

import os

In [11]:
import os
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

image_dir = '/content/images'
if not os.path.exists(image_dir):
    os.makedirs(image_dir)

# Install scikit-image if not already installed
try:
    import skimage.data
    import skimage.io
    import skimage.color # Ensure skimage.color is imported for rgb2gray
except ImportError:
    print("scikit-image not found, installing...")
    !pip install scikit-image
    import skimage.data
    import skimage.io
    import skimage.color

# Load standard images from scikit-image and save them locally
def load_and_save_skimage_image(image_func, filename, target_dir, to_grayscale=False):
    path = os.path.join(target_dir, filename)
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed existing {filename}")

    print(f"Loading {filename} from scikit-image data and saving locally...")
    img_data = image_func()
    if to_grayscale and img_data.ndim == 3: # Convert to grayscale if requested and it's a color image
        img_data = skimage.color.rgb2gray(img_data)
    elif img_data.ndim == 3: # If not converting to grayscale, but it's a color image, ensure it's still treated as such or convert if needed by downstream
        # For images like camera which are already grayscale or lena which might be color
        pass # Keep as is, cv2.imread will handle grayscale conversion if IMREAD_GRAYSCALE is used

    # Convert float image data to uint8 for saving as PNG
    # Handle cases where image_func might return uint8 directly (like camera)
    if img_data.dtype == float:
        img_data_uint8 = (img_data * 255).astype(np.uint8)
    else:
        img_data_uint8 = img_data # Assume it's already uint8 or similar

    skimage.io.imsave(path, img_data_uint8)
    print(f"Saved {filename} to {path}")
    return path

# Using skimage.data.astronaut (converted to grayscale) for the first image
# and skimage.data.camera for the second.
lena_png_path = load_and_save_skimage_image(skimage.data.astronaut, 'lena.png', image_dir, to_grayscale=True) # Renamed to lena.png for compatibility
cameraman_png_path = load_and_save_skimage_image(skimage.data.camera, 'cameraman.png', image_dir, to_grayscale=False)

def create_dummy_image(path, shape=(400, 400), color=0):
    img = Image.new('L', shape, color)
    draw = ImageDraw.Draw(img)
    if 'A.png' in path:
        draw.ellipse((50, 50, 150, 150), fill=255)
        draw.rectangle((100, 100, 200, 200), fill=0)
    elif 'B.png' in path:
        draw.rectangle((150, 150, 250, 250), fill=255)
        draw.ellipse((200, 200, 300, 300), fill=0)
    img.save(path)
    print(f"Created dummy image: {path}")

a_png_path = os.path.join(image_dir, 'A.png')
if not os.path.exists(a_png_path):
    create_dummy_image(a_png_path)

b_png_path = os.path.join(image_dir, 'B.png')
if not os.path.exists(b_png_path):
    create_dummy_image(b_png_path)

OSError: [Errno 30] Read-only file system: '/content'

In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [5]:
def sample_image(image, factor):
    height, width = image.shape[:2]
    sampled_image = cv2.resize(image, (width // factor, height // factor), interpolation=cv2.INTER_NEAREST)
    return sampled_image

def quantize_image(image, levels):
    levels = int(levels)
    if levels <= 0:
        raise ValueError("Number of levels must be greater than 0")
    max_pixel_value = np.max(image)
    min_pixel_value = np.min(image)
    if max_pixel_value == min_pixel_value:
        return np.zeros_like(image, dtype=np.uint8)

    bin_width = (max_pixel_value - min_pixel_value) / levels
    quantized_image = np.floor((image - min_pixel_value) / bin_width) * bin_width + min_pixel_value
    quantized_image = np.clip(quantized_image, 0, 255)
    quantized_image = quantized_image.astype(np.uint8)
    return quantized_image

def plot_images(original, sampled, quantized, title_suffix=""):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(original, cmap='gray')
    plt.title('Original Image' + title_suffix)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(sampled, cmap='gray')
    plt.title('Sampled Image' + title_suffix)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(quantized, cmap='gray')
    plt.title('Quantized Image' + title_suffix)
    plt.axis('off')

    plt.show()

In [6]:
image_path = lena_png_path # Use the new PNG path for Lena
sampling_factor = 14
quantization_levels = 9

# Load image using skimage.io.imread and convert to grayscale if needed
original_image_skimage = skimage.io.imread(image_path)
if original_image_skimage.ndim == 3: # If it's a color image (like astronaut), convert to grayscale
    original_image = skimage.color.rgb2gray(original_image_skimage)
    original_image = (original_image * 255).astype(np.uint8)
else:
    original_image = original_image_skimage.astype(np.uint8) # Ensure it's uint8 if already grayscale

if original_image is None:
    print(f"Error: Unable to load image at {image_path}")
else:
    sampled_image = sample_image(original_image, sampling_factor)
    quantized_image = quantize_image(original_image, quantization_levels)

    plot_images(original_image, sampled_image, quantized_image, f" (Factor={sampling_factor}, Levels={quantization_levels})")

NameError: name 'lena_png_path' is not defined

### Observing Effects by Changing Parameters (Task #1)

In [7]:
sampling_factor_1 = 20
quantization_levels_1 = 4
print(f"Changing parameters: Sampling Factor = {sampling_factor_1}, Quantization Levels = {quantization_levels_1}")
sampled_image_1 = sample_image(original_image, sampling_factor_1)
quantized_image_1 = quantize_image(original_image, quantization_levels_1)
plot_images(original_image, sampled_image_1, quantized_image_1, f" (Factor={sampling_factor_1}, Levels={quantization_levels_1})")

sampling_factor_2 = 5
quantization_levels_2 = 16
print(f"Changing parameters: Sampling Factor = {sampling_factor_2}, Quantization Levels = {quantization_levels_2}")
sampled_image_2 = sample_image(original_image, sampling_factor_2)
quantized_image_2 = quantize_image(original_image, quantization_levels_2)
plot_images(original_image, sampled_image_2, quantized_image_2, f" (Factor={sampling_factor_2}, Levels={quantization_levels_2})")

sampling_factor_3 = 10
quantization_levels_3 = 2
print(f"Changing parameters: Sampling Factor = {sampling_factor_3}, Quantization Levels = {quantization_levels_3}")
sampled_image_3 = sample_image(original_image, sampling_factor_3)
quantized_image_3 = quantize_image(original_image, quantization_levels_3)
plot_images(original_image, sampled_image_3, quantized_image_3, f" (Factor={sampling_factor_3}, Levels={quantization_levels_3})")

Changing parameters: Sampling Factor = 20, Quantization Levels = 4


NameError: name 'original_image' is not defined

## Task #2: Arithmetic and Logical Operations

In [8]:
img1_path = lena_png_path # Use the new PNG path for Lena
img2_path = cameraman_png_path # Use the new PNG path for Cameraman

# Load images using skimage.io.imread
img1_skimage = skimage.io.imread(img1_path)
img2_skimage = skimage.io.imread(img2_path)

# Convert to grayscale and uint8 for consistent processing
if img1_skimage.ndim == 3:
    img1_pil = Image.fromarray((skimage.color.rgb2gray(img1_skimage) * 255).astype(np.uint8))
else:
    img1_pil = Image.fromarray(img1_skimage.astype(np.uint8))

if img2_skimage.ndim == 3:
    img2_pil = Image.fromarray((skimage.color.rgb2gray(img2_skimage) * 255).astype(np.uint8))
else:
    img2_pil = Image.fromarray(img2_skimage.astype(np.uint8))

resize_dims = (256, 256)
img1_resized = img1_pil.resize(resize_dims, Image.Resampling.LANCZOS)
img2_resized = img2_pil.resize(resize_dims, Image.Resampling.LANCZOS)

im1arr = np.asarray(img1_resized, dtype=np.int16)
im2arr = np.asarray(img2_resized, dtype=np.int16)

subtraction = im1arr - im2arr
subtraction_display = np.clip(subtraction, 0, 255).astype(np.uint8)
result_img_sub = Image.fromarray(subtraction_display)
plt.figure(figsize=(6, 6))
plt.imshow(result_img_sub, cmap='gray')
plt.title('Image Subtraction (Lena - Cameraman)')
plt.axis('off')
plt.show()

constant_value = 175
addition_constant = im1arr + constant_value
addition_constant_display = np.clip(addition_constant, 0, 255).astype(np.uint8)
result_img_add_const = Image.fromarray(addition_constant_display)
plt.figure(figsize=(6, 6))
plt.imshow(result_img_add_const, cmap='gray')
plt.title(f'Image Addition (Lena + {constant_value})')
plt.axis('off')
plt.show()

NameError: name 'lena_png_path' is not defined

In [9]:
from PIL import Image

img3_path = os.path.join(image_dir, 'A.png')
img4_path = os.path.join(image_dir, 'B.png')

img3 = Image.open(img3_path).convert('L')
img4 = Image.open(img4_path).convert('L')

resize_dims = (400, 400)
img3 = img3.resize(resize_dims, Image.Resampling.LANCZOS)
img4 = img4.resize(resize_dims, Image.Resampling.LANCZOS)

im3arr = np.asarray(img3, dtype=np.uint8)
im4arr = np.asarray(img4, dtype=np.uint8)

_, im3_binary = cv2.threshold(im3arr, 127, 255, cv2.THRESH_BINARY)
_, im4_binary = cv2.threshold(im4arr, 127, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(im3_binary, cmap='gray')
plt.title('Image A (Binary)')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(im4_binary, cmap='gray')
plt.title('Image B (Binary)')
plt.axis('off')
plt.show()

difference_set = cv2.bitwise_and(im3_binary, cv2.bitwise_not(im4_binary))
result_img_diff = Image.fromarray(difference_set)
plt.figure(figsize=(6, 6))
plt.imshow(result_img_diff, cmap='gray')
plt.title('Set Difference (A - B)')
plt.axis('off')
plt.show()

symmetric_difference = cv2.bitwise_xor(im3_binary, im4_binary)
result_img_symm_diff = Image.fromarray(symmetric_difference)
plt.figure(figsize=(6, 6))
plt.imshow(result_img_symm_diff, cmap='gray')
plt.title('Symmetric Difference (A XOR B)')
plt.axis('off')
plt.show()

intersection = cv2.bitwise_and(im3_binary, im4_binary)
result_img_intersection = Image.fromarray(intersection)
plt.figure(figsize=(6, 6))
plt.imshow(result_img_intersection, cmap='gray')
plt.title('Intersection (A AND B)')
plt.axis('off')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/images/A.png'